# Stage 3 — Micro data

**Friday, 10:50–11:55.**

The same reform, now on SOEP. This notebook is about the pipeline: how prepared survey
data becomes GETTSIM input, what is missing, and how you close the gap.

It stops at GETTSIM's output. Turning that output into a number is analysis, and the
choices it needs — weights, a defensible sample, the right aggregation — are yours.

**In a hurry?** Run All, then start reading at section 6, where your reform goes.

`soep-preparation` does the basic cleaning: negative codes to missings, dtypes,
variables derived from existing ones, renaming. It does not hand you an analysis-ready
dataset — sample selection and the judgement calls that go with it are still yours.

In [ ]:
import json

import numpy as np
import pandas as pd

from gettsim import InputData, MainTarget, TTTargets, copy_environment, main
from gettsim.tt import PiecewisePolynomialParam, TTSIMUnit, get_piecewise_parameters
from soep_preparation.config import BLD, MODULES
from soep_preparation.final_dataset import create_final_dataset

POLICY_DATE = "2023-07-01"
SURVEY_YEAR = 2023

## 1 · What the pipeline produces

Cleaned SOEP modules, a metadata catalogue, and `gettsim_inputs`, which renames the
variables it can to GETTSIM qnames. Details in `soep-preparation/docs/scope.md` and
`soep-preparation/docs/using_with_gettsim.md`.

In [ ]:
gettsim_inputs = pd.read_feather(BLD / "gettsim_inputs" / "gettsim_inputs.arrow")
gettsim_inputs.shape

In [ ]:
list(gettsim_inputs.columns)

The pipeline also emits a mapping report. Read it as *what the pipeline thinks it
covers* — it is scoped to `soep-preparation`'s ambitions, not to your targets. For your
targets, ask GETTSIM, which is the next section.

In [ ]:
report = json.loads(
    (BLD / "gettsim_inputs" / "mapping_report.json").read_text(encoding="utf-8")
)["union"]

print(f"GETTSIM inputs in the mapping: {report['n_inputs_total']}")
print(f"  with a SOEP source:          {report['n_inputs_mapped']}")
print(f"  without:                     {report['n_inputs_unmapped']}")

## 2 · Ask GETTSIM what it needs

Targets first. The leaves are the column names you want back.

In [ ]:
TARGETS = {
    "bürgergeld": {"betrag_m_bg": "bürgergeld_m_bg"},
    "einkommensteuer": {"betrag_m_sn": "einkommensteuer_m_sn"},
    "sozialversicherung": {"beiträge_versicherter_m": "sozialversicherung_m"},
    "kindergeld": {"betrag_m": "kindergeld_m"},
    "wohngeld": {"betrag_m_wthh": "wohngeld_m_wthh"},
}

`MainTarget.templates.input_data_dtypes.tree` returns every input those targets need.
Do not guess this list, and do not read it off the mapping report.

In [ ]:
def qnames(tree, prefix=""):
    """Flatten a nested GETTSIM tree to `a__b__c` qnames."""
    for key, value in tree.items():
        path = f"{prefix}__{key}" if prefix else key
        if isinstance(value, dict):
            yield from qnames(value, path)
        else:
            yield path


def required_inputs(**kwargs):
    template = main(
        main_target=MainTarget.templates.input_data_dtypes.tree,
        policy_date_str=POLICY_DATE,
        tt_targets=TTTargets.tree(TARGETS),
        include_warn_nodes=False,
        **kwargs,
    )
    return sorted(qnames(template))


without_overrides = required_inputs()
len(without_overrides)

That list asks for a contribution history, an Elterngeld biography and an
Arbeitslosengeld claim history per person. SOEP carries none of them.

But most of those inputs exist only to let GETTSIM *compute* five benefit amounts. Hand
it a node whose value you already know and it drops that node's entire upstream subtree
from the template. Five overrides — the two pension benefits, Arbeitslosengeld,
Elterngeld, Unterhaltsvorschuss — buy back most of the list.

In [ ]:
OVERRIDES = {
    "p_id": pd.Series([0]),
    "sozialversicherung": {
        "rente": {
            "altersrente": {"betrag_m": pd.Series([0.0])},
            "erwerbsminderung": {"betrag_m": pd.Series([0.0])},
        },
        "arbeitslosen": {"betrag_m": pd.Series([0.0])},
    },
    "elterngeld": {"betrag_m": pd.Series([0.0])},
    "unterhaltsvorschuss": {"betrag_m": pd.Series([0.0])},
}

needed = required_inputs(input_data=InputData.tree(OVERRIDES))

print(f"inputs without the overrides: {len(without_overrides)}")
print(f"inputs with them:             {len(needed)}")

Those 53 are what the rest of this notebook has to produce: some from the pipeline, some
derived from SOEP variables the pipeline cleans but does not map, and the remainder — the
ones SOEP does not observe at all — as stated constants.

In [ ]:
needed

## 3 · The sample

One survey year, plus six SOEP variables the pipeline cleans but does not map to a
qname. `create_final_dataset` merges the variables you ask for; you name the modules
that hold them. The metadata catalogue
(`soep-preparation/bld/variable_to_metadata_mapping.yaml`) is how you find out which
module a variable lives in.

In [ ]:
EXTRA_VARIABLES = [
    "age",  # pequiv
    "federal_state_of_residence",  # pequiv
    "gesetzliche_rente_y",  # pequiv
    "arbeitslosengeld_y",  # pequiv
    "in_education",  # pgen
    "rented_or_owned",  # hgen
]
EXTRA_MODULES = ["pequiv", "pgen", "hgen"]

extra = create_final_dataset(
    modules={name: MODULES[name].load() for name in EXTRA_MODULES},
    variables=EXTRA_VARIABLES,
    survey_years=[SURVEY_YEAR],
)
extra.shape

Keep only people with a valid interview in `SURVEY_YEAR`. Without this the frame carries
every person ever observed, whose inputs are then silently filled with zeros — which
makes them look poor rather than absent.

In [ ]:
# A few SOEP source modules carry more than one row per person-year; keep the first.
extra = extra.drop_duplicates(subset=["p_id"], keep="first")

soep = gettsim_inputs[gettsim_inputs["survey_year"] == SURVEY_YEAR].merge(
    extra.drop(columns=["hh_id", "hh_id_original"], errors="ignore"),
    on=["p_id", "survey_year"],
    how="inner",
)
soep = (
    soep[soep["age"].notna()]
    .drop_duplicates(subset=["p_id"], keep="first")
    .reset_index(drop=True)
)
print(f"{len(soep):,} people in {soep['hh_id'].nunique():,} households")

Now drop households with an unobserved value in any input that has no stated reading. A
missing value reaching GETTSIM is indistinguishable from a legitimate zero: an
unobserved rent makes a household look rent-free, so ineligible for Wohngeld and lower
in its SGB II Bedarf. We drop the household instead.

The income variables are only required of people old enough to be interviewed — SOEP
does not ask under-16s, so their missing income is an absence, not a gap.

This is a complete-case sample and not a random one. It is where your own cleaning would
do better.

In [ ]:
MUST_BE_OBSERVED = [
    "wohnen__bruttokaltmiete_m_hh",
    "wohnen__heizkosten_m_hh",
    "wohnen__wohnfläche_hh",
    "rented_or_owned",
]
MUST_BE_OBSERVED_IF_INTERVIEWED = [
    "geburtsjahr",
    "einnahmen__bruttolohn_m",
    "einkommensteuer__einkünfte__aus_selbstständiger_arbeit__betrag_y",
]
INTERVIEW_AGE = 16

interviewed = (pd.to_numeric(soep["age"], errors="coerce") >= INTERVIEW_AGE).fillna(True)

incomplete = set()
for column in MUST_BE_OBSERVED:
    incomplete |= set(soep.loc[soep[column].isna(), "hh_id"])
for column in MUST_BE_OBSERVED_IF_INTERVIEWED:
    incomplete |= set(soep.loc[soep[column].isna() & interviewed, "hh_id"])

n_households = soep["hh_id"].nunique()
soep = soep[~soep["hh_id"].isin(incomplete)].reset_index(drop=True)

print(f"dropped {len(incomplete):,} of {n_households:,} households "
      f"({len(incomplete) / n_households:.0%})")
print(f"left with {len(soep):,} people in {soep['hh_id'].nunique():,} households")

## 4 · Derived columns

GETTSIM wants numpy dtypes and no missings. Three helpers, then ordinary pandas.

In [ ]:
def as_int(series, fill=-1):
    """Integer column, missing values as `fill`."""
    return pd.to_numeric(series, errors="coerce").fillna(fill).astype("int64")


def as_float(series):
    """Float column, missing values as 0.0."""
    return pd.to_numeric(series, errors="coerce").fillna(0.0).astype("float64")


def as_bool(series, fill=False):
    """Boolean column, missing values as `fill`."""
    return series.astype("boolean").fillna(fill).astype("bool")

Straight from the pipeline's own GETTSIM mapping, cast and renamed to short column
names. The mapper in section 5 is what pairs them back up with qnames.

In [ ]:
df = pd.DataFrame(index=soep.index)

df["p_id"] = as_int(soep["p_id"])
df["hh_id"] = as_int(soep["hh_id"])
df["geburtsjahr"] = as_int(soep["geburtsjahr"], fill=1970)
df["behinderungsgrad"] = as_int(soep["behinderungsgrad"], fill=0)
df["arbeitsstunden_w"] = as_float(soep["arbeitsstunden_w"])
df["bruttolohn_m"] = as_float(soep["einnahmen__bruttolohn_m"])
df["selbstständig_y"] = as_float(
    soep["einkommensteuer__einkünfte__aus_selbstständiger_arbeit__betrag_y"]
)
df["bruttokaltmiete_m_hh"] = as_float(soep["wohnen__bruttokaltmiete_m_hh"])
df["heizkosten_m_hh"] = as_float(soep["wohnen__heizkosten_m_hh"])
df["wohnfläche_hh"] = as_float(soep["wohnen__wohnfläche_hh"])

Age. SOEP gives the birth month but not the interview date, so the within-year position
of a birthday is approximated by half a year.

In [ ]:
alter = as_int(soep["age"], fill=0)

df["alter"] = alter
df["alter_monate"] = alter * 12 + 6

Family pointers. SOEP gives one parent pointer; the second parent is that parent's
spouse. Defensible for married couples, which is the case the Bedarfsgemeinschaft logic
cares about — unmarried co-resident parents are missed, so their households look like
single-parent households to GETTSIM.

SOEP also records a spouse or parent even when that person did not take part in the
survey year. GETTSIM requires every pointer to resolve to a `p_id` that is present, so a
dangling pointer becomes "no spouse" / "no parent". This shrinks measured
Bedarfsgemeinschaften and is one of the places where sample restriction quietly changes
the answer.

In [ ]:
p_id_ehepartner = as_int(soep["familie__p_id_ehepartner"])
p_id_elternteil_1 = as_int(soep["familie__p_id_elternteil_1"])
p_id_einstandspartner = as_int(soep["bürgergeld__p_id_einstandspartner"])

spouse_of = dict(zip(df["p_id"], p_id_ehepartner, strict=True))
p_id_elternteil_2 = p_id_elternteil_1.map(
    lambda p_id: spouse_of.get(p_id, -1) if p_id >= 0 else -1
).astype("int64")

present = set(df["p_id"])


def valid_pointer(pointer):
    """Pointers to people outside the sample become -1."""
    return pointer.where(pointer.isin(present), -1).astype("int64")


df["p_id_ehepartner"] = valid_pointer(p_id_ehepartner)
df["p_id_elternteil_1"] = valid_pointer(p_id_elternteil_1)
df["p_id_elternteil_2"] = valid_pointer(p_id_elternteil_2)
df["p_id_einstandspartner"] = valid_pointer(p_id_einstandspartner)

# Kindergeld goes to the first parent on record.
df["p_id_kindergeldempfänger"] = df["p_id_elternteil_1"]

Family and education status. A single parent is a parent of a child in the household
with no spouse present.

In [ ]:
has_child = df["p_id"].isin(
    pd.concat([df["p_id_elternteil_1"], df["p_id_elternteil_2"]])
)

df["gemeinsam_veranlagt"] = df["p_id_ehepartner"] >= 0
df["alleinerziehend"] = has_child & (df["p_id_ehepartner"] < 0)
df["in_ausbildung"] = as_bool(soep["in_education"]) & (alter >= 18)

Housing and region. The `federal_state_of_residence` categories are the English labels
`soep-preparation` assigns — check them against the metadata catalogue rather than
assuming German spellings.

In [ ]:
EAST = {
    "Berlin",
    "Brandenburg",
    "Mecklenburg-Vorpommern",
    "Saxony",
    "Saxony-Anhalt",
    "Thuringia",
}

df["bewohnt_eigentum_hh"] = (
    soep["rented_or_owned"].astype("string").eq("Owner").fillna(False).astype("bool")
)
df["wohnort_ost_hh"] = (
    soep["federal_state_of_residence"]
    .astype("string")
    .isin(EAST)
    .fillna(False)
    .astype("bool")
)

The two benefit amounts we take from SOEP rather than let GETTSIM compute. Both are
surveyed annually for the previous year.

In [ ]:
df["altersrente_m"] = as_float(soep["gesetzliche_rente_y"]) / 12
df["arbeitslosengeld_m"] = as_float(soep["arbeitslosengeld_y"]) / 12

GETTSIM requires `*_hh` inputs to be constant within `hh_id`. SOEP household variables
are attached per person and can disagree across members after missing values are filled,
so take the first value each household reports.

In [ ]:
HOUSEHOLD_COLUMNS = [
    "bruttokaltmiete_m_hh",
    "heizkosten_m_hh",
    "wohnfläche_hh",
    "bewohnt_eigentum_hh",
    "wohnort_ost_hh",
]

for column in HOUSEHOLD_COLUMNS:
    df[column] = df.groupby("hh_id")[column].transform("first")

df.head()

## 5 · The mapper

One tree, keyed by qname. Every leaf is either a column of `df` or a constant. Mapped
and assumed sit side by side, and each constant carries the reason it is defensible.

This is the cell to edit. If an assumption below is wrong for your reform, change it
here and rerun.

In [ ]:
MAPPER = {
    # --- identifiers and demographics ---------------------------------------
    "p_id": "p_id",
    "hh_id": "hh_id",
    "alter": "alter",
    "alter_monate": "alter_monate",
    "geburtsjahr": "geburtsjahr",
    "arbeitsstunden_w": "arbeitsstunden_w",
    "behinderungsgrad": "behinderungsgrad",
    # Merkzeichen G is not surveyed; only the Grad der Behinderung is.
    "schwerbehindert_grad_g": False,
    # SOEP surveys wealth only in 2002, 2007, 2012 and 2017, so it is unobserved in
    # 2023. Zero means everyone passes the Vermögensprüfung: this over-states
    # entitlement. If your reform touches the means test, this is the line to fix.
    "vermögen": 0.0,
    # --- family -------------------------------------------------------------
    "familie": {
        "alleinerziehend": "alleinerziehend",
        "p_id_ehepartner": "p_id_ehepartner",
        "p_id_elternteil_1": "p_id_elternteil_1",
        "p_id_elternteil_2": "p_id_elternteil_2",
    },
    # --- housing ------------------------------------------------------------
    "wohnen": {
        "bruttokaltmiete_m_hh": "bruttokaltmiete_m_hh",
        "heizkosten_m_hh": "heizkosten_m_hh",
        "wohnfläche_hh": "wohnfläche_hh",
        "bewohnt_eigentum_hh": "bewohnt_eigentum_hh",
    },
    "wohnort_ost_hh": "wohnort_ost_hh",
    "wohngeld": {
        # SOEP has no Gemeindekennziffer, so the statutory Mietstufe cannot be
        # assigned. 3 is the modal Stufe. This one matters — see the Wohnkosten talk.
        "mietstufe_hh": 3,
    },
    # --- Bürgergeld ---------------------------------------------------------
    "bürgergeld": {
        "p_id_einstandspartner": "p_id_einstandspartner",
        # Decides which Vermögensfreibetrag applies. Does not bind here, because
        # `vermögen` is zero for everyone.
        "bezug_im_vorjahr": True,
    },
    "kindergeld": {
        "p_id_empfänger": "p_id_kindergeldempfänger",
        "in_ausbildung": "in_ausbildung",
    },
    # --- income -------------------------------------------------------------
    "einnahmen": {
        "bruttolohn_m": "bruttolohn_m",
        # Household-level and top-coded in SOEP; below the Sparerpauschbetrag for
        # most Bürgergeld households anyway.
        "kapitalerträge_y": 0.0,
        "renten": {
            "aus_berufsständischen_versicherungen_m": 0.0,  # rare
            "basisrente_m": 0.0,  # not separable in SOEP
            "betriebliche_altersvorsorge_m": 0.0,  # not separable in SOEP
            "geförderte_private_vorsorge_m": 0.0,  # not separable in SOEP
            "sonstige_private_vorsorge_m": 0.0,  # not separable in SOEP
        },
    },
    "einkommensteuer": {
        "gemeinsam_veranlagt": "gemeinsam_veranlagt",
        "abzüge": {
            # Riester/Rürup contributions not separable in SOEP.
            "beitrag_private_rentenversicherung_m": 0.0,
            # Childcare costs are surveyed irregularly; zero understates deductions.
            "kinderbetreuungskosten_m": 0.0,
            "p_id_kinderbetreuungskostenträger": -1,  # follows from the line above
        },
        "einkünfte": {
            "aus_selbstständiger_arbeit": {"betrag_y": "selbstständig_y"},
            # Negligible for the working-age population Bürgergeld concerns.
            "aus_forst_und_landwirtschaft": {"betrag_y": 0.0},
            # SOEP self-employment income is not split by Einkunftsart; booked as
            # selbstständige Arbeit instead.
            "aus_gewerbebetrieb": {"betrag_y": 0.0},
            "aus_nichtselbstständiger_arbeit": {
                # Not surveyed; GETTSIM applies the Arbeitnehmerpauschbetrag.
                "tatsächliche_werbungskosten_y": 0.0,
            },
            # Rental income is surveyed at household level only.
            "aus_vermietung_und_verpachtung": {"betrag_y": 0.0},
            # Affects health-insurance treatment only; rare in this population.
            "ist_hauptberuflich_selbstständig": False,
            "sonstige": {
                "alle_weiteren_y": 0.0,  # residual category
                "rente": {
                    # Only binds when private pension income is non-zero, which it
                    # is not here.
                    "alter_beginn_leistungsbezug_sonstige_private_vorsorge": 67,
                },
            },
        },
    },
    # Maintenance received is not reliably measured in SOEP.
    "unterhalt": {"tatsächlich_erhaltener_betrag_m": 0.0},
    # --- the five overridden nodes ------------------------------------------
    # Take-up is not observed.
    "unterhaltsvorschuss": {"betrag_m": 0.0},
    # Eligibility needs a birth history and prior net income; out of scope for a
    # Bürgergeld reform.
    "elterngeld": {"betrag_m": 0.0},
    "sozialversicherung": {
        # Private insurance is rare among Bürgergeld-eligible households.
        "kranken": {"beitrag": {"privat_versichert": False}},
        # Avoids the childless surcharge; refine if your reform is sensitive to it.
        "pflege": {"beitrag": {"hat_kinder": True}},
        "arbeitslosen": {"betrag_m": "arbeitslosengeld_m"},
        "rente": {
            "bezieht_rente": False,  # set together with the overridden amount
            # Far future, so no one is treated as retired.
            "jahr_renteneintritt": 2080,
            "altersrente": {"betrag_m": "altersrente_m"},
            # Disability pension receipt is not cleanly identified in SOEP.
            "erwerbsminderung": {"betrag_m": 0.0},
            # Grundrente needs a full contribution history.
            "grundrente": {"grundrentenzeiten_monate": 0},
        },
    },
}

The mapper should cover exactly the inputs section 2 said were needed. Check rather than
trust — this fires if GETTSIM's requirements shift under you.

In [ ]:
supplied = set(qnames(MAPPER))
missing = sorted(set(needed) - supplied)
spurious = sorted(supplied - set(needed))

print(f"missing:  {missing}")
print(f"spurious: {spurious}")

## 6 · Running the reform

Same two environments as yesterday. The cell below is the only one you need to change:
paste your own reform between the markers, or leave the demo reform in place.

In [ ]:
status_quo = main(
    main_target=MainTarget.policy_environment, policy_date_str=POLICY_DATE
)

In [ ]:
# ---------------------------- YOUR REFORM GOES HERE ----------------------------
# Paste the reform you built yesterday. If Thursday did not finish, leave this
# cell exactly as it is and work with the demo reform.

reform = copy_environment(status_quo)
reform["bürgergeld"]["parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg"] = (
    PiecewisePolynomialParam(
        value=get_piecewise_parameters(
            func_type="piecewise_linear",
            parameter_list=[
                {"interval": "(-inf, 0)", "intercept": 0, "slope": 0},
                {"interval": "[0, 100)", "slope": 1.0},
                {"interval": "[100, 520)", "slope": 0.2},
                {"interval": "[520, 1000)", "slope": 0.15},
                {"interval": "[1000, 1200)", "slope": 0.1},
                {"interval": "[1200, inf)", "slope": 0.0},
            ],
            leaf_name="parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg",
            xnp=np,
        ),
        input_unit=TTSIMUnit.EUR.PER_MONTH,
        output_unit=TTSIMUnit.EUR.PER_MONTH,
    )
)
# -------------------------------------------------------------------------------

`InputData.df_and_mapper` hands GETTSIM the frame and the mapper together;
`MainTarget.results.df_with_mapper` returns the targets under the names you gave them.

In [ ]:
def run(policy_environment):
    return main(
        main_target=MainTarget.results.df_with_mapper,
        policy_date_str=POLICY_DATE,
        policy_environment=policy_environment,
        input_data=InputData.df_and_mapper(df=df, mapper=MAPPER),
        tt_targets=TTTargets.tree(TARGETS),
        include_warn_nodes=False,
    )


before = run(status_quo)
after = run(reform)
before.head()

Most people are unaffected — they are not on Bürgergeld, or their earnings are outside
the band the reform touches. Look at the ones that move.

In [ ]:
comparison = pd.DataFrame(
    {
        "status quo": before["bürgergeld_m_bg"],
        "reform": after["bürgergeld_m_bg"],
    }
)
comparison["difference"] = comparison["reform"] - comparison["status quo"]

changed = comparison[comparison["difference"].abs() > 1e-9]
print(f"{len(changed):,} of {len(comparison):,} people see a different Bürgergeld")
changed.head(15)

## 7 · One trap before you aggregate

For the persona every aggregation level coincided. On SOEP they do not: a household can
hold more than one Bedarfsgemeinschaft, more than one Steuernummer. A `_bg` value is
repeated on every member of the Bedarfsgemeinschaft, so summing it across people
double-counts.

Ask GETTSIM for the group ids and take one value per group before you sum anything.

In [ ]:
group_ids = main(
    main_target=MainTarget.results.df_with_mapper,
    policy_date_str=POLICY_DATE,
    input_data=InputData.df_and_mapper(df=df, mapper=MAPPER),
    tt_targets=TTTargets.tree({"bg_id": "bg_id", "hh_id": "hh_id"}),
    include_warn_nodes=False,
)

per_person = pd.DataFrame(
    {
        "hh_id": group_ids["hh_id"],
        "bg_id": group_ids["bg_id"],
        "bürgergeld_m_bg": before["bürgergeld_m_bg"],
    }
)

naive = per_person["bürgergeld_m_bg"].sum()
correct = per_person.drop_duplicates(subset=["bg_id"])["bürgergeld_m_bg"].sum()

print(f"summed over people:              {naive:>12,.0f} € per month")
print(f"summed over Bedarfsgemeinschaften: {correct:>10,.0f} € per month")
print(f"the naive sum is {naive / correct:.1f}× too large")

---

That is where this notebook stops: GETTSIM output on real households, not results.

To turn it into a number you still need the household weight (`hh_weighting_factor` in
the pipeline — SOEP is a sample, and these weights are not rescaled after the households
we dropped), a sample you are willing to defend rather than the complete cases we kept,
and a view on every constant in the mapper. Those are your choices, and they are most of
the work.